# サーブノート 解析ノートブック

管理用スプレッドシートの「**解析待ち**」の行を順番に処理し、お手本動画と比較した採点結果を書き戻します。

**使い方**：上から順に ▶ を押して実行してください（2回目以降は ③ の URL を書き換える必要はありません）。

| セル | やること |
|---|---|
| ① 準備 | 解析コードを GitHub から取得し、MediaPipe を入れる |
| ② ログイン | スプレッドシートと Drive を使う許可を出す |
| ③ 設定 | 管理用スプレッドシートを開く |
| ④ 解析待ちの確認 | 「解析待ち」の行を一覧で表示する |
| ⑤ 道具の準備 | 動画の取得・変換・スプシ書き込みの関数を用意する |
| ⑥ 解析の実行 | 解析待ちを1件ずつ処理して、結果をスプシに書く |

In [ ]:
# ① 準備：解析コードを GitHub から取得し、必要なライブラリを入れる
#   - 解析のロジックは backend/ai/ にある（アプリと同じコードを使い回す）
#   - お手本動画（success.mp4）と MediaPipe のモデルもリポジトリに入っている
REPO_URL = "https://github.com/saya-38812/softtennis.git"
BRANCH = "feature/google-workflow"

!rm -rf /content/softtennis
!git clone -q --depth 1 -b {BRANCH} {REPO_URL} /content/softtennis
!pip install -q mediapipe

import sys
# 「from ai.xxx import …」で backend/ai/ のコードを読み込めるようにする
sys.path.insert(0, "/content/softtennis/backend")
print("準備完了")

In [ ]:
# ② ログイン：Colab を使っている自分（運営者）のアカウントで、スプシと Drive を使えるようにする
#   ポップアップが出たら、運営者のアカウントを選んで「許可」を押す
from google.colab import auth
auth.authenticate_user()

import google.auth
import gspread
from googleapiclient.discovery import build

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",  # スプシの読み書き
    "https://www.googleapis.com/auth/drive",          # 依頼者から共有された動画の読み込み
]
creds, _ = google.auth.default(scopes=SCOPES)
gc = gspread.authorize(creds)                              # スプシ用
drive = build("drive", "v3", credentials=creds)            # Drive 用
print("ログイン完了")

In [ ]:
# ③ 設定：管理用スプレッドシートの URL を貼り付ける
SPREADSHEET_URL = "ここに管理用スプレッドシートのURLを貼る"
SHEET_NAME = "sessions"

sheet = gc.open_by_url(SPREADSHEET_URL).worksheet(SHEET_NAME)

# 見出し行から「列名 → 何列目か」の対応表を作る
# （列の順番をコードに直接書かないので、スプシの列が増えても壊れにくい）
header = sheet.row_values(1)
COL = {name: i + 1 for i, name in enumerate(header)}
print("開いたシート:", sheet.title, "／ 列:", header)

In [ ]:
# ④ 解析待ちの確認
import pandas as pd

STATUS_PENDING = "解析待ち"

def fetch_pending():
    """スプシを読み直して、status が「解析待ち」の行だけを返す（_row は何行目か）"""
    records = sheet.get_all_records()  # 見出し行を除いた全行（2行目から始まる）
    rows = [dict(r, _row=i + 2) for i, r in enumerate(records)]
    return [r for r in rows if r.get("status") == STATUS_PENDING]

pending = fetch_pending()
if pending:
    display(pd.DataFrame(pending)[["_row", "received_at", "requester_email", "mode", "serve_count", "video_name"]])
else:
    print("解析待ちはありません")

In [ ]:
# ⑤ 道具の準備：動画の取得・変換・スプシへの書き込み
import datetime
import io
import json
import os
import subprocess
from zoneinfo import ZoneInfo

from googleapiclient.http import MediaIoBaseDownload

WORK_DIR = "/content/work"
os.makedirs(WORK_DIR, exist_ok=True)


def now_jst():
    """日本時間の現在時刻（Colab のサーバーは世界標準時なので、明示的に日本時間にする）"""
    return datetime.datetime.now(ZoneInfo("Asia/Tokyo")).strftime("%Y-%m-%d %H:%M:%S")


def download_video(file_id, dest_path):
    """Drive のファイルIDを指定して動画をダウンロードする（大きい動画は10MBずつ分けて受け取る）"""
    request = drive.files().get_media(fileId=file_id)
    with io.FileIO(dest_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request, chunksize=10 * 1024 * 1024)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return dest_path


def to_mp4(src_path):
    """
    どんな動画も mp4（H.264）に変換する。
    ブラウザで録画した webm はフレーム数などの情報が欠けていて、OpenCV で正しく扱えないことがあるため。
    """
    dst_path = os.path.splitext(src_path)[0] + "_conv.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", src_path,
         "-c:v", "libx264", "-preset", "veryfast", "-an", dst_path],
        check=True,
    )
    return dst_path


def update_row(row_number, **values):
    """指定した行の、指定した列だけを書き換える（例：update_row(5, status="完了")）"""
    cells = [gspread.Cell(row_number, COL[key], value) for key, value in values.items()]
    sheet.update_cells(cells)

print("道具の準備完了")

In [ ]:
# ⑥ 解析の実行：解析待ちを1件ずつ処理する
import logging
logging.getLogger().setLevel(logging.WARNING)  # MediaPipe の細かいログを抑える

from ai.session_runner import analyze_session

pending = fetch_pending()  # 実行する直前に読み直す（④ のあとに新しい依頼が来ているかもしれないので）
print(f"解析待ち: {len(pending)}件")

for r in pending:
    row = r["_row"]
    print(f"\n▶ {row}行目：{r['requester_email']} / {r['mode']} / {r['serve_count']}本")
    # 先に「解析中」にしておく（途中で止まったときに、どこまで進んだかわかるように）
    update_row(row, status="解析中", error_message="")
    files = []
    try:
        src = download_video(r["drive_file_id"], f"{WORK_DIR}/{r['session_id']}_src")
        files.append(src)
        mp4 = to_mp4(src)
        files.append(mp4)

        serves = json.loads(r.get("serves_json") or "[]")
        result = analyze_session(mp4, r["mode"], serves)

        update_row(
            row,
            status="完了",
            analyzed_at=now_jst(),
            result_json=json.dumps(result, ensure_ascii=False),
        )
        print("  ✅ 完了:", result["summary"])
    except Exception as e:
        # 1件で失敗しても、ほかの依頼の解析は続ける
        update_row(row, status="エラー", analyzed_at=now_jst(), error_message=str(e))
        print("  ❌ エラー:", e)
    finally:
        # ダウンロードした動画は消す（Colab のディスク容量を圧迫しないように）
        for path in files:
            if os.path.exists(path):
                os.remove(path)

print("\nすべて終わりました")